# W&B Sweep Report

Queries wandb sweep results and builds a summary table + Report. Self-contained: does not depend on `criteo_functions.py` or the other notebooks.

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import wandb
import polars as pl
import polars.selectors as cs
import pandas as pd
import wandb_workspaces.reports.v2 as wr
import altair as alt
from wandb_helper import *
from ipydatagrid import DataGrid
pl.Config.set_tbl_rows(100)

## Download Action

In [ ]:
# dl_all = get_project_runs()

In [ ]:
#dl_all = get_project_runs(project_name="criteo_boost")

In [ ]:
# 'system_metrics.system.proc.memory.rssMB
# 'system_metrics.system.gpu.0.memoryAllocatedBytes')/(1024**3)

In [ ]:
df.shape

In [ ]:
pl.DataFrame(data={"a":[{"1": 1,"2":2, }, {"3": 3,"4":4}]},schema={"a": pl.List(pl.Struct({"key": pl.String, "value": pl.Int16}))})

In [ ]:
api = wandb.Api(timeout=60)
runs = list(api.runs("seanv507/criteo_boost"))

## Process the runs

In [ ]:
criteo_boost_df = make_runs_df(dl_criteo_boost)

In [ ]:
criteo_boost_df = convert_int_double_strs(criteo_boost_df)

In [ ]:
criteo_boost_df=pl.read_parquet("data/criteo_boost.parquet")

In [ ]:
criteo_boost_df = pl.read_parquet("data/criteo_boost_01.parquet")

In [ ]:
criteo_boost_df = (
    criteo_boost_df
    .with_columns(
        pl.col("feature_importance").list.eval(pl.element().struct[0].str.contains("encode")).list.any().alias("config.target_encode")
    ).with_columns( 
        pl.when(pl.col("config.target_encode") | (pl.col("created_at")<= pl.datetime(2026,8,1,time_zone="UTC"))).then(pl.lit("target"))
        .when((pl.col("config.learner.gradient_booster.tree_train_param.max_cat_to_onehot")==4294967295)).then(pl.lit("ohe"))
        .when(pl.col("project")=="criteo_boost_enum").then(pl.lit("node"))
        .otherwise(pl.col("config.learner.gradient_booster.tree_train_param.cat_order")).alias("config.cat_order_fix")
    )
    .sort("summary_metrics.epoch")
    .drop("config.n_estimators_est", strict=False)
    .join_asof(
        pl.DataFrame({"config.n_estimators_est": [50,100,150,200,300,600,1000]}),
        left_on="summary_metrics.epoch",
        right_on="config.n_estimators_est",strategy="forward"
    )
)

In [ ]:
#df.write_parquet("data/criteo_boost_01.parquet")

### duplicate_cols

In [ ]:
duplicate_cols = ["config.gamma",
                  "config.learner.gradient_booster.tree_train_param.gamma", #use min_split_loss
                  "config.max_bin",
                  "config.cat_order",
                  'config.learner.gradient_booster.tree_train_param.cat_order', # cat_order_fix
                  "config.cat_reg_lambda",
                  "config.max_depth", 
                  "config.reg_lambda", 
                  "config.learning_rate",
                  "config.min_child_weight",
                  "config.subsample",
                  "config.max_cat_threshold",
                  "config.target_encode",
                  ]

In [ ]:
# the config.values are on the rare occasions that 
# a sweep was used and the values were first defined in a config dict and passed to xgboost
# this means that they are undefined most of the time.

ignore_cols = [
    'config.device',
     'config.enable_categorical',
    'config.learner.metrics',
    'config.learner.objective.name',
    'config.learner.objective.reg_loss_param.scale_pos_weight',
    'config.learner.generic_param.seed',
    #'config.learner.generic_param.device',
    'config.learner.generic_param.n_jobs',
    'config.learner.generic_param.nthread',
    'config.learner.generic_param.rng_state',
    'config.learner.generic_param.random_state',
    'config.learner.generic_param.seed_per_iteration',
    'config.learner.generic_param.validate_parameters',
    'config.learner.generic_param.fail_on_invalid_gpu_id',
    'config.learner.gradient_booster.name',
    'config.learner.gradient_booster.updater',
    'config.learner.gradient_booster.dart_train_param.one_drop',
    'config.learner.gradient_booster.dart_train_param.rate_drop',
    'config.learner.gradient_booster.dart_train_param.skip_drop',
    'config.learner.gradient_booster.dart_train_param.sample_type',
    'config.learner.gradient_booster.dart_train_param.normalize_type',
    'config.learner.gradient_booster.tree_train_param.eta',
    #'config.learner.gradient_booster.tree_train_param.learning_rate',
    'config.learner.gradient_booster.tree_train_param.alpha', #have not used it None/0 variability
    'config.learner.gradient_booster.tree_train_param.reg_alpha',
    'config.learner.gradient_booster.tree_train_param.lambda',
    #'config.learner.gradient_booster.tree_train_param.reg_lambda',
    #'config.learner.gradient_booster.tree_train_param.max_bin',
    #'config.learner.gradient_booster.tree_train_param.max_depth',
    #'config.learner.gradient_booster.tree_train_param.subsample',
    'config.learner.gradient_booster.tree_train_param.max_leaves', #none/0 variability
    'config.learner.gradient_booster.tree_train_param.grow_policy',
    'config.learner.gradient_booster.tree_train_param.refresh_leaf',
    'config.learner.gradient_booster.tree_train_param.max_delta_step',
    #'config.learner.gradient_booster.tree_train_param.cat_order',
    #'config.learner.gradient_booster.tree_train_param.cat_reg_lambda',
    #'config.learner.gradient_booster.tree_train_param.min_split_loss',
    #'config.learner.gradient_booster.tree_train_param.min_child_weight',
    #'config.learner.gradient_booster.tree_train_param.max_cat_threshold',
    #'config.learner.gradient_booster.tree_train_param.max_cat_to_onehot',
    'config.learner.gradient_booster.tree_train_param.sampling_method',
    'config.learner.gradient_booster.tree_train_param.colsample_bynode',
    'config.learner.gradient_booster.tree_train_param.colsample_bytree',
    'config.learner.gradient_booster.tree_train_param.sparse_threshold',
    'config.learner.gradient_booster.tree_train_param.colsample_bylevel',
    'config.learner.gradient_booster.tree_train_param.monotone_constraints',
    'config.learner.gradient_booster.tree_train_param.interaction_constraints',
    'config.learner.gradient_booster.specified_updater',
    #'config.learner.gradient_booster.gbtree_model_param.num_trees',
    'config.learner.gradient_booster.gbtree_model_param.num_parallel_tree',
    'config.learner.gradient_booster.gbtree_train_param.updater',
    'config.learner.gradient_booster.gbtree_train_param.tree_method',
    'config.learner.gradient_booster.gbtree_train_param.updater_seq',
    'config.learner.gradient_booster.gbtree_train_param.process_type',
    'config.learner.learner_model_param.num_class',
    'config.learner.learner_model_param.base_score',
    'config.learner.learner_model_param.num_target',
    'config.learner.learner_model_param.num_feature', # not hyperparam
    'config.learner.learner_model_param.boost_from_average',
    'config.learner.learner_train_param.booster',
    'config.learner.learner_train_param.num_class',
    'config.learner.learner_train_param.objective',
    'config.learner.learner_train_param.base_score',
    'config.learner.learner_train_param.num_target',
    'config.learner.learner_train_param.multi_strategy',
    'config.learner.learner_train_param.boost_from_average',
    'config.learner.learner_train_param.disable_default_eval_metric',
    'config.version',
    #'config.cat_cutoff',
    'config.tree_method',
    'config.n_estimators',
    #'config.sample_every',
    #'config.cat_pseudo_count',
    'config.early_stopping_rounds',
    'config.data_dir',
    'config.association_rules']


In [ ]:
config_variability = (
    criteo_boost_df
    .drop(duplicate_cols + ignore_cols).rename(replace_strings)
    .select(cs.starts_with("config.").value_counts().implode().list.eval(pl.element().struct.rename_fields(["value","count"])))
)
    
config_variability.glimpse()

In [ ]:
config_variables = [
"sample_every",
"subsample",
"n_estimators_est",
"max_bin",
"max_depth",
"min_split_loss",
"reg_lambda",
"learning_rate",
"min_child_weight",
"cat_cutoff",
"cat_order_fix",
"cat_reg_lambda",
"cat_pseudo_count",
"max_cat_threshold",
"max_cat_to_onehot",
]
# cat_order

# num_feature
# learner.generic_param.device                <list[struct[2]]> [{'value': 'cuda:0', 'count': 144}, {'value': None, 'count': 25}, {'value': 'cpu', 'count': 182}]
# learner.gradient_booster.gbtree_model_para… <list[struct[2]]> [{'value': 0, 'count': 326}, {'value': None, 'count': 25}]
# variables that 
#"reg_alpha",  [{'value': 0, 'count': 326}, {'value': None, 'count': 25}]
# max_leaves                                  <list[struct[2]]> [{'value': None, 'count': 25}, {'value': 0, 'count': 32
# (default =0) 

Missing n_estimators
enable_categorical

In [ ]:
config_variability.write_excel("config_variability.xlsx")

In [ ]:
plot_df = (
    criteo_boost_df
    .group_by("config.sample_every", "project", "config.cat_order_fix")
    .agg(pl.col('summary_metrics.validation_1-logloss.min').min(), pl.len().alias("count"))
    .sort("config.sample_every", 'summary_metrics.validation_1-logloss.min')
)
print(plot_df)
(
    alt.Chart(plot_df)
    .mark_bar(clip=True)
    .encode(
        x=alt.X("project:N",sort=None), 
        y=alt.Y(r"summary_metrics\.validation_1-logloss\.min", 
                scale=alt.Scale(domain=[.43, .48])),
        color=r"config\.sample_every:N")

)

In [ ]:
plot_df = (
    criteo_boost_df
    #.filter(pl.col("config.n_estimators_est")== 150)
    .group_by("config.sample_every")
    .agg(pl.col('summary_metrics.validation_1-logloss.min').min())
    .with_columns(pl.col("config.sample_every").pow(.5).alias("sample_every^(0.5)"))
)
base_chart = (
    alt.Chart(plot_df)
    .encode(
        x=alt.X(r"sample_every^(0\.5)"), 
        y=alt.Y(r"summary_metrics\.validation_1-logloss\.min", 
                scale=alt.Scale(domain=[.43, .48])),
    )

)
    
chart =  base_chart.mark_line() + base_chart.mark_point() + base_chart.transform_regression(r"sample_every^(0\.5)",r"summary_metrics\.validation_1-logloss\.min").mark_line()
chart

We would expect the error to scale linearly with $\sqrt{\text{ sample\_every }}$, but this doesn't seem to happen.  Possibly because of the wrong parameter settings at each sample_every?

In [ ]:
from IPython.display import display, Markdown

with pl.Config() as cfg:
    cfg.set_tbl_formatting('ASCII_MARKDOWN')
    display(Markdown(repr(plot_df)))

In [ ]:
criteo_boost_gpu_categorical = criteo_boost_df.filter(
    (pl.col("project")=="criteo_boost_gpu_categorical") & 
    (pl.col("config.learner.gradient_booster.tree_train_param.max_cat_to_onehot")==10) &
    (pl.col("state")=="finished") & 
    (pl.col("config.sample_every")==1)
)

OHE is now enabled by default with tree_train_param.max_cat_to_onehot set to MAX_INT
so we filter that out

We also have sample every = 100 (around 30 runs)

In [ ]:

agg_cols = ["cat_cutoff", "cat_order_fix", ]
plot_df =(
    criteo_boost_gpu_categorical
    .filter(pl.col("config.n_estimators_est")==150)
    .drop(duplicate_cols)
    .rename(replace_strings)
    .rename(lambda x: x.replace("config.",""))
    .group_by(agg_cols)
    .agg(pl.len(), pl.col('validation log loss').min())
    .sort(["cat_cutoff", 'validation log loss'])
)
base_chart = alt.Chart(plot_df.drop_nulls()).encode(
    x="cat_cutoff", 
    y=alt.Y("validation log loss:Q", scale=alt.Scale(domain=[.44,.45])),
    color = "cat_order_fix:N",)

base_chart.mark_line(clip=True) + base_chart.mark_point(clip=True)

can't seem to get node below 0.4492 for node, whereas 0.44872 for 200, and would expect to reach ?0.4485?

also why does error increase on 75?

In [ ]:
plot_df

what parameter settings for small error on node, 200

In [ ]:
(
    criteo_boost_df
    .select(cs.starts_with("config.") | cs.starts_with("summary_"))
    .drop(duplicate_cols + ignore_cols)
    .rename(replace_strings)
    .filter((pl.col("config.cat_order_fix")=='node') & (pl.col("config.cat_cutoff")==200))
).glimpse()

using tree order reduces error by ~ 0.06. Also the cat_cutoff slope is higher for tree.

In [ ]:
with pl.Config() as cfg:
    cfg.set_tbl_formatting('ASCII_MARKDOWN')
    display(Markdown(repr(plot_df)))
    

# Breakdown


## Categorical Ordering - effect of cat_cutoff Preprocessing/Tree/Node

Identify categorical data processing

In [ ]:
# "2026-10-09T15:21:51Z
# "%Y-%m-%dT%H:%M%:%SZ"
gp_cols = [pl.col("created_at").dt.date().alias("created_at_date"), "project", "state",
           "config.cat_order_fix", "config.max_cat_to_onehot","config.cat_cutoff","config.cat_pseudo_count",
           ]
sort_cols = ["created_at_date", "project",  "state", "config.cat_order_fix","config.max_cat_to_onehot","config.cat_cutoff","config.cat_pseudo_count", ]
plot_df = (
    criteo_boost_df
    .drop(duplicate_cols + ignore_cols)
    .rename(replace_strings)
    
    .group_by(gp_cols)
    .agg(pl.col("validation log loss").min(), pl.len().alias("count"))
    .rename(replace_strings)
    .sort(sort_cols)
)
plot_df

In [ ]:
plot_df.write_excel("encodings.xlsx")

In [ ]:
## Categorical Ordering - effect of cat_cutoff

In [ ]:
plot_df = (
    criteo_boost_df
    .drop(duplicate_cols)
    .rename(replace_strings)
    .filter(pl.col("config.sample_every")==1)
    .group_by("config.cat_order_fix","config.cat_cutoff")
    .agg(pl.col("validation log loss").min())
    .sort(["config.cat_order_fix","config.cat_cutoff"])
)

(alt.Chart(plot_df.rename(lambda x: x.replace("config.","")))
.mark_point()
.encode(
    x=alt.X("cat_cutoff",scale=alt.Scale(type="log")),
    y=alt.Y("validation log loss").scale(zero=False),
    color = "cat_order_fix"
)
).properties(width=800, height=800)

In [ ]:
zz = criteo_boost_df.drop(duplicate_cols+ignore_cols).rename(replace_strings).select(
    pl.col(["config."+c for c in config_variables]),"validation log loss").filter(pl.col("config.cat_cutoff")==200).sort("config.cat_order_fix")

In [ ]:
DataGrid(zz.to_pandas().T)

In [ ]:
zz.write_excel("data/cutoff_100")

In [ ]:
(criteo_boost_df
    .drop(duplicate_cols)
    .rename(replace_strings)
    .filter(pl.col("cat_order")=="tree")
)["max_cat_to_onehot"].value_counts()

In [ ]:
cat_cutoff_dfs = criteo_boost_df.drop_nulls("summary_metrics.validation_1-logloss.min").drop(cs.ends_with("cat_order")).partition_by(
    ["config.cat_cutoff", 
     'config.cat_order_fix',], as_dict=True)
cat_cutoff_dfs.keys()

In [ ]:
zz = (
    criteo_boost_df
    .select(
        cs.starts_with("config.")
        .value_counts(sort=True)
        .implode()
    ).drop(duplicate_cols)
    .rename(replace_strings)
)

In [ ]:
config_values

In [ ]:
cat_cutoff_df

In [ ]:

for key, cat_cutoff_df in cat_cutoff_dfs.items():
    # if key[0] is None or key[1] is None:
    #     continue
    key_str = f"cat_cutoff: {key[0]} cat_order: {key[1]}"
    cat_cutoff_df = (
        cat_cutoff_dfs[key]
        .drop(duplicate_cols + ignore_cols, strict=False)
        .rename(replace_strings)
        .drop_nulls(subset = "validation log loss")
    )
    # if len(cat_cutoff_df)==0:
    #     continue
    #'config.learner.gradient_booster.tree_train_param.gamma' = min_split_loss
    config_values = (
        cat_cutoff_df
        .select(
            cs.starts_with("config.").exclude("config.learner.generic_param.device")
            .value_counts(sort=True)
            .implode()
        )
    )
    not_unique = config_values.select(pl.all().list.len()>1).row(0,named=True)
    not_unique = [k for k, v  in not_unique.items() if v ]
    if not not_unique:
        print(f"No variation for {key_str}")
        continue
    baseline = (
        config_values
        .select(pl.col(not_unique).list[0].struct[0])
        .rename(replace_strings) # in struct
        .rename(lambda x: x.replace("config.", ""))
        .row(0, named=True)
    )
    grid = (
        cat_cutoff_df.rename(lambda x: x.replace("config.", ""))
        .group_by(baseline.keys()).agg("run_id",pl.col("validation log loss").mean(),pl.len()).sort(baseline.keys())
    )
    slices = one_at_a_time(grid, list(baseline.keys()))
    slices  = slices.with_columns(
        pl.concat_str(
            [pl.format(f"{f}: {{}}", pl.col(f)) for f in baseline.keys()],
            separator=", ",ignore_nulls=True
        ).alias("fixed params")
    )
    if len(slices) == 0:
        print(f"No variation for {key_str}")
        continue
    plot_dfs = {}

    for varying in slices["varying"].value_counts(sort=True)['varying']:

        plot_df = (
            slices.filter(pl.col("varying")==varying)
            .drop(varying)
            .rename({"values":varying})
            .explode(varying, "run_id", "validation log loss")
        )
        plot_dfs[varying] = plot_df
    ranges = {v: (d["validation log loss"].min(), d["validation log loss"].max()) for v, d in plot_dfs.items()}
    span = 1.1 * max(hi - lo for lo, hi in ranges.values())   # 10% padding
    best = min(lo for lo, _ in ranges.values())
    charts = {}
    for varying, plot_df in plot_dfs.items():
        lo, hi = ranges[varying]
        mid = (lo + hi) / 2
        base = alt.Chart(plot_df, title=varying).encode(
            x=f"{varying}:Q",
            y=alt.Y(
                "validation log loss:Q",
                #scale=alt.Scale(zero=False, domain=[.44,.48])
                scale=alt.Scale(domain=[mid - span / 2, mid + span / 2], nice=False, zero=False),
            ),
            color=alt.Color(
                "fixed params",
                legend=alt.Legend(labelLimit=0, labelExpr="split(datum.label, ', ')", symbolOffset=0, rowPadding=8),
            ),
        ).properties(width=350, height=300)
        best_rule = alt.Chart().mark_rule(strokeDash=[4, 4], color="gray").encode(y=alt.datum(best))
        charts[varying] = base.mark_line(clip=True) + base.mark_point(clip=True) + best_rule

    panel = (
        alt.concat(*charts.values(), columns=2, title = key_str).resolve_scale(color="independent", y="independent")
    )
    panel.display()
    panel.save(f"figures/{key_str}.svg")

## OHE vs cat cutoff

In [ ]:
criteo_boost_ohe = (
    criteo_boost_df
    .filter((pl.col("config.cat_order_fix")=="ohe") & (pl.col("config.sample_every")==1))
)
zz =(
    criteo_boost_ohe
    .drop(duplicate_cols+ignore_cols)
    .rename(replace_strings).rename(lambda x:x.replace('config.',''))
    .group_by(config_variables).agg(pl.col("project"), pl.len(), pl.col("validation log loss").min())
).sort("validation log loss")

DataGrid(zz.to_pandas())

In [ ]:
zz.columns

In [ ]:
plot_df = (
    criteo_boost_ohe
    .drop(duplicate_cols+ignore_cols)
    .rename(replace_strings).rename(lambda x:x.replace('config.',''))
    .group_by(["cat_cutoff", "n_estimators_est"]).agg(pl.col("project"), pl.len(), pl.col("validation log loss").min())
)
base_chart = alt.Chart(plot_df).encode(x=alt.X("cat_cutoff").scale(type="log"), y=alt.Y("validation log loss").scale(zero=False), color="n_estimators_est:N")
base_chart.mark_point() + base_chart.mark_line()

To get the actual runs for one slice, for example to plot logloss against the varying parameter, join the fixed values back to the run data:



```{python}
row = slices.row(0, named=True)
v = row["varying"]
fixed = {p: row[p] for p in baseline if p != v}
runs = cat_cutoff_dfs[(500, "node")].filter(**fixed).sort(v)
```
filter(**kwargs) works as an equality filter here, even with the dotted column names, because they're passed as dict keys.

In [ ]:
[c for c in  cat_cutoff_dfs[(500, "node")].columns if "gamma" in c]

```
run = api.run('seanv507/criteo_boost_sweep_04/auwure38')
```

In [ ]:
ps = get_project_sweeps("seanv507")

In [ ]:
project_sweeps = make_sweeps_df(ps)

In [ ]:
project_sweeps["sweep_id"].unique()

In [ ]:

project_sweeps.null_count()

In [ ]:
project_sweeps.head()

In [ ]:
project_sweeps.with_columns(pl.col('sweep_config').struct['parameters']).unnest('parameters')

In [ ]:
a1 = project_sweeps.explode('sweeps').unnest('sweeps').explode('sweep_runs').unnest('sweep_runs')


In [ ]:
project_sweeps_e = project_sweeps.explode("sweep_ids")
project_sweeps_e.head()

In [ ]:
project_sweeps_e.group_by(pl.col("project").str.slice(0,10)).len().sort("len",descending=True)

In [ ]:

# 3. Log the derived table to a dedicated analysis run.
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    name=f"{SWEEP_ID}-column-statistics",
    job_type="analysis",
) as analysis_run:
    analysis_run.log({
        TABLE_KEY: wandb.Table(dataframe=stats)
    })
    analysis_run_id = analysis_run.id

# 4. Create a Report containing the table.
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="Criteo boost sweep statistics",
    description=f"Column statistics across all runs in sweep {SWEEP_ID}.",
    blocks=[
        wr.H2("Sweep column statistics"),
        wr.PanelGrid(
            runsets=[
                wr.Runset(
                    entity=ENTITY,
                    project=PROJECT,
                    filters=[
                        wr.Metric("name").isin([analysis_run_id])
                    ],
                )
            ],
            panels=[
                wr.WeavePanelSummaryTable(
                    table_name=TABLE_KEY,
                    layout=wr.Layout(w=24, h=12),
                )
            ],
        ),
    ],
)

report.save()
print(report.url)



> **⚠️ Duplicate** — this cell redoes the entire pipeline from the `ENTITY = ...` cell above through the Report-building cell, self-contained in one cell. Same missing-`pandas`-import issue noted above applies here too.

In [ ]:

ENTITY = "seanv507"
PROJECT = "criteo_boost_sweep"
SWEEP_ID = "s15xvple"
TABLE_KEY = "sweep_column_statistics"

# 1. Read the 45 sweep runs.
api = wandb.Api()
sweep = api.sweep(f"{ENTITY}/{PROJECT}/{SWEEP_ID}")

rows = []
for run in sweep.runs:
    summary = dict(run.summary)
    rows.append({
        "learning_rate": run.config.get("learning_rate"),
        "max_depth": run.config.get("max_depth"),
        "subsample": run.config.get("subsample"),
        # This sweep stores the objective's minimum under this summary key.
        "validation_1-logloss": summary.get(
            "validation_1-logloss.min"
        ),
    })

# 2. Calculate one row per column.
values = pd.DataFrame(rows).apply(pd.to_numeric, errors="coerce")

stats = (
    values.agg(["min", "max", "mean"])
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

# 3. Log the derived table to a dedicated analysis run.
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    name=f"{SWEEP_ID}-column-statistics",
    job_type="analysis",
) as analysis_run:
    analysis_run.log({
        TABLE_KEY: wandb.Table(dataframe=stats)
    })
    analysis_run_id = analysis_run.id

# 4. Create a Report containing the table.
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="Criteo boost sweep statistics",
    description=f"Column statistics across all runs in sweep {SWEEP_ID}.",
    blocks=[
        wr.H2("Sweep column statistics"),
        wr.PanelGrid(
            runsets=[
                wr.Runset(
                    entity=ENTITY,
                    project=PROJECT,
                    filters=[
                        wr.Metric("name").isin([analysis_run_id])
                    ],
                )
            ],
            panels=[
                wr.WeavePanelSummaryTable(
                    table_name=TABLE_KEY,
                    layout=wr.Layout(w=24, h=12),
                )
            ],
        ),
    ],
)

report.save()
print(report.url)
